# Solutions — Bakehouse Sales Analysis (Medium 02)

**Dataset:** `samples.bakehouse.sales_transactions`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

transactions = spark.table("samples.bakehouse.sales_transactions")

## Solution 1 — Daily Revenue & Transaction Count

In [ ]:
result_1 = (
    transactions
    .withColumn("date", F.to_date("dateTime"))
    .groupBy("date")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("totalPrice"), 2).alias("total_revenue")
    )
    .orderBy("date")
)

## Solution 2 — Revenue & Quantity per Product with Rank

In [ ]:
# Why: window over the whole table (no partitionBy) ranks all products globally
w = Window.orderBy(F.col("total_revenue").desc())
result_2 = (
    transactions
    .groupBy("product")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("total_revenue"),
        F.sum("quantity").alias("total_quantity")
    )
    .withColumn("revenue_rank", F.rank().over(w))
)

## Solution 3 — Running Total Revenue

In [ ]:
# Why: rowsBetween with unbounded preceding = cumulative sum over all prior rows
w = Window.orderBy("dateTime").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result_3 = (
    transactions
    .select(
        "transactionID","dateTime","totalPrice",
        F.round(F.sum("totalPrice").over(w), 2).alias("running_revenue")
    )
)

## Solution 4 — Top 3 Products per Payment Method

In [ ]:
# Why: partitionBy paymentMethod so rank resets per method
w = Window.partitionBy("paymentMethod").orderBy(F.col("total_revenue").desc())
result_4 = (
    transactions
    .groupBy("paymentMethod","product")
    .agg(F.round(F.sum("totalPrice"), 2).alias("total_revenue"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") <= 3)
    .orderBy("paymentMethod","rank")
)

## Solution 5 — Month-over-Month Revenue (Lag)

In [ ]:
# Why: lag(1) over ordered months fetches previous month's revenue
monthly = (
    transactions
    .withColumn("year", F.year("dateTime"))
    .withColumn("month", F.month("dateTime"))
    .groupBy("year","month")
    .agg(F.round(F.sum("totalPrice"), 2).alias("revenue"))
    .orderBy("year","month")
)
w = Window.orderBy("year","month")
result_5 = monthly.withColumn("prev_month_revenue", F.lag("revenue", 1).over(w))

## Solution 6 — Transactions Above Avg Quantity per Product

In [ ]:
# Why: window avg per product lets us compare each row against its product's avg
w = Window.partitionBy("product")
result_6 = (
    transactions
    .withColumn("avg_product_quantity", F.avg("quantity").over(w))
    .filter(F.col("quantity") > F.col("avg_product_quantity"))
    .select("transactionID","product","quantity","avg_product_quantity")
)

## Solution 7 — Revenue Percentage per Product

In [ ]:
# Why: global sum via window (no partitionBy) allows ratio calculation per row
w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
result_7 = (
    transactions
    .groupBy("product")
    .agg(F.round(F.sum("totalPrice"), 2).alias("total_revenue"))
    .withColumn(
        "revenue_pct",
        F.round(F.col("total_revenue") / F.sum("total_revenue").over(w) * 100, 2)
    )
    .orderBy(F.col("revenue_pct").desc())
)